# Create Sample Incoming Batch

This notebook creates a small sample batch from locally available NCBI phenotype and genotype tables so the AMR system can simulate newly incoming data.

## 1. Configuration

Update the two source paths below if your local NCBI files are stored outside the default `data/` folder.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent

# Update these paths if your local files are stored elsewhere.
PHENOTYPE_PATH = PROJECT_ROOT / 'data' / 'phenotype.tsv'
GENOTYPE_PATH = PROJECT_ROOT / 'data' / 'genotype.tsv'

BATCH_NAME = 'batch_001'
N_BIOSAMPLES = 50
RANDOM_SEED = 42

OUTPUT_DIR = PROJECT_ROOT / 'data' / 'incoming' / BATCH_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT :', PROJECT_ROOT)
print('PHENOTYPE_PATH:', PHENOTYPE_PATH)
print('GENOTYPE_PATH :', GENOTYPE_PATH)
print('OUTPUT_DIR    :', OUTPUT_DIR)

if not PHENOTYPE_PATH.exists():
    raise FileNotFoundError(f'Phenotype file not found: {PHENOTYPE_PATH}')
if not GENOTYPE_PATH.exists():
    raise FileNotFoundError(f'Genotype file not found: {GENOTYPE_PATH}')

## 2. Load Raw NCBI Tables

In [ ]:
phenotype = pd.read_csv(
    PHENOTYPE_PATH,
    sep='\t',
    dtype={
        '#BioSample': 'string',
        'Antibiotic': 'string',
        'Resistance phenotype': 'string',
        'Measurement sign': 'string',
        'MIC (mg/L)': 'string',
    },
).rename(columns={'#BioSample': 'BioSample'})

genotype = pd.read_csv(
    GENOTYPE_PATH,
    sep='\t',
    dtype={
        '#BioSample': 'string',
        'Element symbol': 'string',
        'Subtype': 'string',
        'Class': 'string',
        'Subclass': 'string',
        'Method': 'string',
        '% Coverage of reference': 'string',
        '% Identity to reference': 'string',
    },
).rename(columns={'#BioSample': 'BioSample'})

phenotype['BioSample'] = phenotype['BioSample'].str.strip()
genotype['BioSample'] = genotype['BioSample'].str.strip()

print('phenotype shape:', phenotype.shape)
print('genotype shape :', genotype.shape)
display(phenotype.head())
display(genotype.head())

## 3. Sample Shared Isolates

To simulate incoming AMR records, we sample `BioSample` IDs that appear in both tables.

In [ ]:
phenotype_isolates = set(phenotype['BioSample'].dropna().unique().tolist())
genotype_isolates = set(genotype['BioSample'].dropna().unique().tolist())
shared_isolates = sorted(phenotype_isolates & genotype_isolates)

if not shared_isolates:
    raise ValueError('No shared BioSample IDs were found between phenotype and genotype tables.')

sample_size = min(N_BIOSAMPLES, len(shared_isolates))
sampled_biosamples = (
    pd.Series(shared_isolates)
    .sample(n=sample_size, random_state=RANDOM_SEED, replace=False)
    .sort_values()
    .tolist()
)

batch_phenotype = phenotype[phenotype['BioSample'].isin(sampled_biosamples)].copy()
batch_genotype = genotype[genotype['BioSample'].isin(sampled_biosamples)].copy()

manifest = pd.DataFrame({'BioSample': sampled_biosamples})

print('shared isolates total :', len(shared_isolates))
print('sampled isolates      :', len(sampled_biosamples))
print('batch phenotype rows  :', len(batch_phenotype))
print('batch genotype rows   :', len(batch_genotype))
display(manifest.head())

## 4. Save Incoming Batch Files

In [ ]:
phenotype_out = OUTPUT_DIR / 'phenotype_incoming.tsv'
genotype_out = OUTPUT_DIR / 'genotype_incoming.tsv'
manifest_out = OUTPUT_DIR / 'biosample_manifest.csv'
summary_out = OUTPUT_DIR / 'batch_summary.json'

batch_phenotype.to_csv(phenotype_out, sep='\t', index=False)
batch_genotype.to_csv(genotype_out, sep='\t', index=False)
manifest.to_csv(manifest_out, index=False)

summary = {
    'batch_name': BATCH_NAME,
    'sample_size': int(len(sampled_biosamples)),
    'phenotype_rows': int(len(batch_phenotype)),
    'genotype_rows': int(len(batch_genotype)),
    'unique_antibiotics': sorted(batch_phenotype['Antibiotic'].dropna().astype(str).unique().tolist()),
    'source_phenotype_path': str(PHENOTYPE_PATH),
    'source_genotype_path': str(GENOTYPE_PATH),
}

summary_out.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('Saved files:')
print('-', phenotype_out)
print('-', genotype_out)
print('-', manifest_out)
print('-', summary_out)

display(batch_phenotype.head())
display(batch_genotype.head())

## 5. Optional Quick Summary

In [ ]:
antibiotic_summary = (
    batch_phenotype.groupby('Antibiotic', dropna=False)
    .agg(
        rows=('BioSample', 'size'),
        biosamples=('BioSample', 'nunique'),
        resistant=('Resistance phenotype', lambda s: (s.astype(str).str.lower() == 'resistant').sum()),
        susceptible=('Resistance phenotype', lambda s: (s.astype(str).str.lower() == 'susceptible').sum()),
    )
    .sort_values(['rows', 'biosamples'], ascending=[False, False])
    .reset_index()
)

display(antibiotic_summary)